# PKG Fraud signals v1 — typology screening on the customer metrics

**Payment Knowledge Graph (PKG) · Treasury Management · Data Science**

Screens Treasury Management entities for five simple suspicious-activity structures and writes three small tables
for the PKG Insights app. Every signal is a **pattern worth review**, not evidence of wrongdoing: each typology has
common legitimate explanations, which the app lists next to the evidence.

| Typology | What it looks for | Built from |
|---|---|---|
| **Pass-through** | Money in ≈ money out and little of it stays (needs the deposit balance); repeated over months | monthly metrics + deposit balances |
| **Funnel** | Many payers in, few payees out; small payments in, large payments out | monthly metrics |
| **Split payments** | Payments clustered just below a reporting threshold; several such payments to one counterparty on one day | daily edges |
| **Activity spike** | A quiet account suddenly moves far more money, mostly with counterparties it never dealt with | monthly metrics |
| **Round-trip** | Money paid out comes back from the same counterparties (2-node cycles in the payment graph) | counterparty-pair table |

**Guards against the usual false positives:** self-transfers and payments to the client's own accounts at other banks
are excluded from pass-through and round-trip; payments within the same corporate family are excluded from round-trip;
payment-processor-like entities (thousands of payers or payees) and industries where pass-through is normal (payroll,
payment processing, title and escrow, property management, freight brokerage, staffing, insurance agencies,
securities brokerage) are down-weighted and tagged; scores are percentiles **within peers of the same industry and size**.

**Writes** (small, external parquet in `bdahd01p_dlcdi1_cdi_tm`): `pkg_fraud_signals` (flagged entities × last 3 months,
scores and evidence), `pkg_fraud_history` (12 months of metrics and balances for flagged entities), `pkg_fraud_evidence`
(near-threshold payment days and round-trip counterparty pairs). Scanning the daily edges and the pair table is
too heavy for an app; these tables keep it fast. Re-run monthly after the metrics build.

### Pre-registered expectations — checked in §7

| # | Expectation |
|---|---|
| E1 | Each typology flags fewer than 2% of screened entities (looser means the gates need tightening) |
| E2 | Fewer than 30% of flagged entities are processor-like or in pass-through industries (more means the guards are not biting) |
| E3 | At least one entity matches two or more typologies |

In [ ]:
# =====================================================================
# 0 · CONFIG
# =====================================================================
TGT_DB      = "bdahd01p_dlcdi1_cdi_tm"
MET_TABLE   = f"{TGT_DB}.pkg_cust_metrics_monthly"
PAIR_TABLE  = f"{TGT_DB}.pkg_cust_pair_month"
EDGE_DAILY  = f"{TGT_DB}.pkg_edge_daily"
CUST_DIM    = f"{TGT_DB}.pkg_cust_dim"
CPTY_DIM    = f"{TGT_DB}.pkg_cpty_dim"
CPTY_NAME_COL, CPTY_FI_COL = "cpty_name", "fi"
DEP_TABLE   = "dsihd01p_dsi.lap_dsi_universe_optimized"   # daily balances; None = score pass-through without balances
OUT_PREFIX  = "pkg_fraud_"                                # → pkg_fraud_signals, pkg_fraud_history, pkg_fraud_evidence
OUT_DIR     = "../metrics/fraud/v1"
SHUFFLE_PARTITIONS = 800

END_MONTH   = None      # None = latest month in the metrics table
N_SCORE     = 3         # scoring months (latest and the two before), so the app can show persistence
HIST_MONTHS = 12        # months of history kept for flagged entities (also the spike baseline)
POPULATION  = "tm"      # "tm" = entities with a customer power id; "all" = every customer
MIN_FLOW_3M = 300_000   # screened only if money in + out over 3 months reaches this
MIN_PEER    = 100       # smallest peer group (industry × size band) before falling back to a broader industry

# ── typology gates (a typology is matched only when every gate passes) ──
PT_MIN, PT_MONTH_MIN, PT_MONTHS_REQ = 0.90, 0.85, 2   # pass-through: 3-month ratio, monthly ratio, months at that ratio
PT_MAX_DAYS_HELD = 5.0                                 # …and the average balance holds ≤ 5 days of inflow (balance required)
FUN_MIN_PAYERS, FUN_MIN_FAN, FUN_MIN_TOP1_OUT, FUN_MIN_PT = 20, 10.0, 0.50, 0.70
SPLIT_THRESHOLDS = [10_000]   # add 3_000 to also test the funds-transfer record-keeping threshold   # reporting-type thresholds; 3,000 (funds-transfer recordkeeping) can be added
SPLIT_BAND, SPLIT_MIN_BELOW, SPLIT_MIN_RATIO = 0.10, 10, 3.0  # band = 10% under / over; ≥ 10 payments under; ≥ 3× more under than over
SPLIT_MIN_Z = 4.0             # …and the excess below is significant: (under − over) / √(under + over) ≥ 4
SPLIT_MIN_DAYS = 3            # …and on ≥ 3 days several under-threshold payments to / from one counterparty add up to more than the threshold
SPIKE_MIN_FLOW_M, SPIKE_MIN_X, SPIKE_MIN_NEW, SPIKE_BASE_FLOOR = 250_000, 10.0, 0.50, 1_000
RT_MIN_AMT, RT_MIN_SHARE, RT_MIN_PAIR = 250_000, 0.25, 10_000

# ── guards ───────────────────────────────────────────────────────────
PROCESSOR_MIN_CPTY = 2_000    # ≥ this many payers or payees a month on average → processor-like
PASS_THROUGH_NAICS = ["5223", "541214", "541191", "5313", "488510", "5613", "5242", "5231"]
DISCOUNT_PROCESSOR, DISCOUNT_INDUSTRY = 0.5, 0.6      # multiply pass-through and funnel scores (both when both hold → 0.3)
TIER_HIGH, TIER_ELEVATED = 70, 50

In [ ]:
# =====================================================================
# 1 · SETUP — shims, session, helpers
# =====================================================================
import numpy as np, warnings
for _a, _t in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _a): setattr(np, _a, _t)
if "bool" not in np.__dict__: np.bool = np.bool_
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"): pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):    pd.Series.iteritems = pd.Series.items
for _c in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_c)

import time, pathlib, html as _html
from functools import reduce
from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
DISK = StorageLevel.DISK_ONLY
OUT = pathlib.Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time(); CHECKS = []

def fmt_df(df, pct=(), money=()):
    d = df.copy()
    for c in d.columns:
        s = d[c]
        if c in pct:    d[c] = s.map(lambda v: "—" if pd.isna(v) else f"{v:.1%}")
        elif c in money: d[c] = s.map(lambda v: "—" if pd.isna(v) else f"${v:,.0f}")
        elif pd.api.types.is_integer_dtype(s): d[c] = s.map(lambda v: f"{v:,}")
        elif pd.api.types.is_float_dtype(s):   d[c] = s.map(lambda v: "—" if pd.isna(v) else f"{v:,.3f}")
    return d

class Block:
    """Collects every table and note of a cell and displays them once, as one image-friendly output."""
    def __init__(self, title):
        self.h = [f"<div class='tex2jax_ignore mathjax_ignore' style='font-family:Arial,sans-serif;font-size:12px'>"
                  f"<h3 style='margin:2px 0 6px'>{title}</h3>"]
    def table(self, df, title, pct=(), money=(), save=None, max_rows=60):
        if save: df.to_csv(OUT / f"{save}.csv", index=False)
        d = fmt_df(df.head(max_rows), pct, money)
        self.h.append(f"<p style='margin:8px 0 2px'><b>{title}</b> <span style='color:#777'>[{len(df):,} rows]</span></p>" +
                      d.to_html(index=False, border=0).replace("<table", "<table style='border-collapse:collapse;font-size:11px' cellpadding='3'"))
        return self
    def kv(self, pairs, title):
        return self.table(pd.DataFrame([{"item": k, "value": v if isinstance(v, str) else f"{v:,}" if isinstance(v, (int, np.integer))
                                         else f"{v:,.4f}" if isinstance(v, float) else str(v)} for k, v in pairs]), title)
    def note(self, s):
        self.h.append(f"<p style='margin:4px 0;color:#333'>{s}</p>"); return self
    def check(self, item, status, detail):
        CHECKS.append({"item": item, "status": status, "detail": detail})
        col = {"READY": "#2e7d32", "FIX": "#e65100", "BLOCKER": "#b71c1c", "INFO": "#555"}[status]
        return self.note(f"<span style='color:{col};font-weight:bold'>[{status}]</span> {_html.escape(item)}: {_html.escape(detail)}")
    def show(self, t0=None):
        if t0: self.note(f"<span style='color:#999'>wall {time.time() - t0:,.0f}s</span>")
        display(HTML("".join(self.h) + "</div>"))

def topd(df):
    for c, t in df.dtypes:
        if t.startswith("decimal"): df = df.withColumn(c, F.col(c).cast("double"))
    return df.toPandas()

def savefig(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=120, bbox_inches="tight"); plt.close(fig)

def ym_idx(ym): y, m = map(int, ym[:7].split("-")); return y * 12 + m - 1
def ym_of(i):   return f"{i // 12:04d}-{i % 12 + 1:02d}"
def ym_add(ym, k): return ym_of(ym_idx(ym) + k)
def t_col(c):   return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - 1).cast("int")
def norm_pwr(c):
    """Power id as a comparable string: trimmed, blank → null, leading zeros removed (deposit and metrics ids differ in padding)."""
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(F.regexp_replace(s, "^0+(?=.)", ""))
def safe_div(a, b): return F.when(b > 0, a / b)

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {TGT_DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
def write_full(df, name, n_files=8):
    """Small non-partitioned external parquet table: rewrite files, then drop + re-create so the schema always matches."""
    path = f"{DB_LOC}/{name}"
    df.repartition(n_files).write.mode("overwrite").option("compression", "snappy").parquet(path)
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in df.schema.fields)
    spark.sql(f"DROP TABLE IF EXISTS {TGT_DB}.{name}")
    spark.sql(f"CREATE EXTERNAL TABLE {TGT_DB}.{name} ({cols}) STORED AS PARQUET LOCATION '{path}'")
    spark.catalog.refreshTable(f"{TGT_DB}.{name}")
print(f"Spark {spark.version} | output {TGT_DB} → {DB_LOC} | charts and CSVs → {OUT.resolve()}")

## §1 Inputs and the scoring window

The latest month in the metrics table is the scoring month unless `END_MONTH` is set. Twelve months are read: three
scoring months, each with a 3-month window, and a 6-month baseline before the window for the activity spike.

In [ ]:
# =====================================================================
# 1 · INPUTS — tables, months, columns                              [OUTPUT BLOCK 1]
# =====================================================================
t0 = time.time(); B = Block("§1 · Inputs and the scoring window")
def months_of(table):
    try:
        return sorted(r[0].split("=", 1)[1] for r in spark.sql(f"SHOW PARTITIONS {table}").collect())
    except Exception:
        return sorted(r[0] for r in spark.table(table).select("month").distinct().collect())
MET_MONTHS = months_of(MET_TABLE)
END = END_MONTH or MET_MONTHS[-1]
M_HIST = [ym_add(END, -k) for k in range(HIST_MONTHS - 1, -1, -1)]              # 12 months ending END
M_SCORE = M_HIST[-N_SCORE:]                                                    # scoring months
M_WIN = M_HIST[-(N_SCORE + 2):]                                                # months any 3-month window touches
missing = [m for m in M_HIST if m not in MET_MONTHS]
if missing: B.check("metrics months", "FIX" if missing[0] < M_WIN[0] else "BLOCKER", f"missing {missing}")
if [m for m in M_WIN if m not in MET_MONTHS]: B.show(); raise RuntimeError(f"metrics months missing inside the scoring windows: {missing}")

MET = spark.table(MET_TABLE).filter(F.col("month").isin(M_HIST))
CORE = ["amt_all_in", "amt_all_out", "ntxn_all_in", "ntxn_all_out", "ncpty_all_in", "ncpty_all_out"]
OPT = ["amt_ext_in", "amt_int_in", "top1_share_out", "top1_share_in", "selfpay_amt_out", "new_amt_share_in", "new_amt_share_out",
       "share_WIRE_out", "share_WIRE_in", "active_days_in", "active_days_out", "n_fi_out", "n_fi_in", "avg_ticket_in", "avg_ticket_out"]
miss_core = [c for c in CORE + ["mdm_id", "month"] if c not in MET.columns]
if miss_core: B.check("metric columns", "BLOCKER", f"missing {miss_core}"); B.show(); raise RuntimeError(f"metrics lack {miss_core}")
OPT_HAVE = [c for c in OPT if c in MET.columns]
B.check("metric columns", "READY" if len(OPT_HAVE) == len(OPT) else "INFO",
        "all present" if len(OPT_HAVE) == len(OPT) else f"optional columns absent: {[c for c in OPT if c not in OPT_HAVE]}")

CD = spark.table(CUST_DIM)
ID_COLS = [c for c in ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc",
                       "naics2", "naics_status", "is_tm", "customer_start_dt"] if c in CD.columns]
try: PAIR_COLS = spark.table(PAIR_TABLE).columns; PAIR_OK = {"cust", "dir", "other", "amount"} <= set(PAIR_COLS)
except Exception: PAIR_OK, PAIR_COLS = False, []
try: ED_COLS = spark.table(EDGE_DAILY).columns; ED_OK = True
except Exception: ED_OK, ED_COLS = False, []
DEP_OK = False
if DEP_TABLE:
    try:
        DEP_COLS = spark.table(DEP_TABLE).columns
        DEP_OK = all(c in DEP_COLS for c in ["cust_pwr_id", "edw_tda_load_dt", "balance"])
    except Exception: DEP_OK = False
B.kv([("metrics months available", f"{MET_MONTHS[0]} … {MET_MONTHS[-1]} ({len(MET_MONTHS)})"),
      ("scoring months", ", ".join(M_SCORE)), ("months read", f"{M_HIST[0]} … {M_HIST[-1]}"),
      ("population", "entities with a customer power id (Treasury Management)" if POPULATION == "tm" else "all customers"),
      ("screening floor", f"money in + out ≥ ${MIN_FLOW_3M:,} over 3 months")], "1a · Window")
B.check("pair table (round-trip)", "READY" if PAIR_OK else "FIX",
        "available" if PAIR_OK else f"{PAIR_TABLE} unreadable: round-trip is skipped")
B.check("daily edges (split payments)", "READY" if ED_OK else "FIX", "available" if ED_OK else f"{EDGE_DAILY} unreadable: split payments skipped")
B.check("deposit balances (pass-through)", "READY" if DEP_OK else "FIX",
        "available" if DEP_OK else "unavailable: pass-through needs balances and is skipped (money in ≈ money out alone describes most businesses)")
B.show(t0)

## §2 Entity × month grid, balances, rolling windows

Every screened entity gets all twelve months; a month without a metrics row is a month with no payments (zero), so a
dormant account looks dormant. Balances are the monthly average of the daily total across the client's accounts.

In [ ]:
# =====================================================================
# 2 · GRID, BALANCES, WINDOW FEATURES                               [OUTPUT BLOCK 2]
# =====================================================================
t0 = time.time(); B = Block("§2 · Entity × month grid and window features")
M0 = ym_idx(M_HIST[0])
ident = CD.select(*ID_COLS)
if POPULATION == "tm":
    ident = ident.filter(F.col("customer_pwr_id").isNotNull() & (F.trim(F.col("customer_pwr_id").cast("string")) != ""))
ident = ident.dropDuplicates(["mdm_id"])
mvals = MET.select("mdm_id", "month", *[F.col(c).cast("double").alias(c) for c in CORE + OPT_HAVE])
ents = mvals.select("mdm_id").distinct().join(ident.select("mdm_id"), "mdm_id")
months_df = spark.createDataFrame([(m, ym_idx(m) - M0) for m in M_HIST], "month string, t int")
G = (ents.crossJoin(F.broadcast(months_df)).join(mvals, ["mdm_id", "month"], "left").join(ident, "mdm_id", "left"))
ZERO = [c for c in CORE + ["amt_ext_in", "amt_int_in", "selfpay_amt_out", "active_days_in", "active_days_out"] if c in CORE + OPT_HAVE]
for c in ZERO: G = G.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))
for c in ["top1_share_out", "top1_share_in", "new_amt_share_in", "new_amt_share_out", "share_WIRE_out", "share_WIRE_in",
          "selfpay_amt_out", "amt_ext_in", "n_fi_out", "n_fi_in", "active_days_in"]:
    if c not in G.columns: G = G.withColumn(c, F.lit(None).cast("double"))
G = G.withColumn("pwr", norm_pwr("customer_pwr_id"))

# ── balances: monthly average of the daily total across the client's accounts (deposit book only) ──
if DEP_OK:
    d0, d1 = f"{M_WIN[0]}-01", f"{ym_add(END, 1)}-01"
    dep = (spark.table(DEP_TABLE).filter((F.col("edw_tda_load_dt") >= F.lit(d0)) & (F.col("edw_tda_load_dt") < F.lit(d1)))
                .select(norm_pwr("cust_pwr_id").alias("pwr"), F.to_date("edw_tda_load_dt").alias("d"), F.col("balance").cast("double").alias("b"))
                .filter("pwr is not null"))
    BAL = (dep.groupBy("pwr", "d").agg(F.sum("b").alias("b")).withColumn("month", F.date_format("d", "yyyy-MM"))
              .groupBy("pwr", "month").agg(F.avg(F.greatest("b", F.lit(0.0))).alias("bal_avg")))
    G = G.join(BAL, ["pwr", "month"], "left")
else:
    G = G.withColumn("bal_avg", F.lit(None).cast("double"))

w3 = Window.partitionBy("mdm_id").orderBy("t").rangeBetween(-2, 0)
wb = Window.partitionBy("mdm_id").orderBy("t").rangeBetween(-7, -2)          # 6-month baseline before the spike month's window
wpre = Window.partitionBy("mdm_id").orderBy("t").rangeBetween(Window.unboundedPreceding, -8)
s3 = lambda c: F.sum(c).over(w3)
G = (G.withColumn("out_adj", F.greatest(F.col("amt_all_out") - F.coalesce(F.col("selfpay_amt_out"), F.lit(0.0)), F.lit(0.0)))
      .withColumn("pt_m", safe_div(F.least("amt_all_in", "out_adj"), F.greatest("amt_all_in", "out_adj")))
      .withColumn("flow_m", F.col("amt_all_in") + F.col("amt_all_out"))
      .withColumn("in_3m", s3("amt_all_in")).withColumn("out_3m", s3("out_adj")).withColumn("out_raw_3m", s3("amt_all_out"))
      .withColumn("flow_3m", F.col("in_3m") + F.col("out_raw_3m"))
      .withColumn("pt_3m", safe_div(F.least("in_3m", "out_3m"), F.greatest("in_3m", "out_3m")))
      .withColumn("pt_months", F.sum(F.when(F.col("pt_m") >= PT_MONTH_MIN, 1).otherwise(0)).over(w3))
      .withColumn("bal_3m", F.avg("bal_avg").over(w3))
      .withColumn("days_held", safe_div(F.col("bal_3m"), F.col("in_3m") / 91.0))
      .withColumn("turnover", safe_div(F.col("in_3m"), F.col("bal_3m")))
      .withColumn("payers_avg", F.avg("ncpty_all_in").over(w3)).withColumn("payees_avg", F.avg("ncpty_all_out").over(w3))
      .withColumn("fan", F.col("payers_avg") / F.greatest(F.col("payees_avg"), F.lit(1.0)))
      .withColumn("top1_out", F.avg("top1_share_out").over(w3))
      .withColumn("ticket_in", safe_div(F.col("in_3m"), s3("ntxn_all_in")))
      .withColumn("ticket_out", safe_div(F.col("out_raw_3m"), s3("ntxn_all_out")))
      .withColumn("ticket_ratio", safe_div(F.col("ticket_out"), F.col("ticket_in")))
      .withColumn("wire_out", safe_div(s3(F.col("amt_all_out") * F.coalesce(F.col("share_WIRE_out"), F.lit(0.0))), F.col("out_raw_3m")))
      .withColumn("new_payer_share", safe_div(s3(F.col("amt_all_in") * F.coalesce(F.col("new_amt_share_in"), F.lit(0.0))), F.col("in_3m")))
      .withColumn("ext_in_share", safe_div(s3(F.coalesce(F.col("amt_ext_in"), F.lit(0.0))), F.col("in_3m")))
      .withColumn("selfpay_share_3m", safe_div(s3(F.coalesce(F.col("selfpay_amt_out"), F.lit(0.0))), F.col("out_raw_3m")))
      .withColumn("base", F.expr("percentile_approx(flow_m, 0.5)").over(wb))
      .withColumn("base_n", F.count("flow_m").over(wb))
      .withColumn("pre_flow", F.sum("flow_m").over(wpre))
      .withColumn("new_share_m", safe_div(F.col("amt_all_in") * F.coalesce(F.col("new_amt_share_in"), F.lit(0.0))
                                          + F.col("amt_all_out") * F.coalesce(F.col("new_amt_share_out"), F.lit(0.0)), F.col("flow_m")))
      .withColumn("spike_x", F.col("flow_m") / F.greatest(F.coalesce(F.col("base"), F.lit(0.0)), F.lit(float(SPIKE_BASE_FLOOR)))))
start_ok = (F.col("customer_start_dt").cast("date") < F.add_months(F.to_date(F.concat("month", F.lit("-01"))), -12)) \
    if "customer_start_dt" in G.columns else F.lit(False)
G = G.withColumn("existed_before", (F.coalesce(F.col("base"), F.lit(0.0)) >= SPIKE_BASE_FLOOR) | (F.coalesce(F.col("pre_flow"), F.lit(0.0)) > 0)
                 | F.coalesce(start_ok, F.lit(False)))
G = G.persist(DISK)
g = G.agg(F.countDistinct("mdm_id").alias("entities"), F.count("*").alias("rows"),
          F.sum((F.col("bal_avg").isNotNull()).cast("int")).alias("with_balance")).first()
SC = G.filter(F.col("month").isin(M_SCORE) & (F.col("flow_3m") >= MIN_FLOW_3M)).persist(DISK)
sc = SC.groupBy("month").agg(F.count("*").alias("screened"), F.sum(F.col("bal_3m").isNotNull().cast("int")).alias("with_balance"),
                             F.sum("flow_3m").alias("flow_3m")).orderBy("month")
B.kv([("entities in the population with any payment in 12 months", g["entities"]), ("entity-months in the grid", g["rows"]),
      ("entity-months with a deposit balance", f"{g['with_balance']:,} ({g['with_balance'] / max(g['rows'], 1):.1%})")], "2a · Grid")
B.table(topd(sc), "2b · Entities screened per scoring month (above the activity floor)", money=("flow_3m",), save="f1_2b_screened")
B.show(t0)

## §3 Split payments (the electronic analogue of structuring)

Classic structuring is about **cash** kept under the $10,000 currency-reporting threshold; this data has no cash. What it
can show is the electronic analogue: payments that cluster just **below** a threshold far more than just **above** it,
and days when several such payments to or from one counterparty add up to more than the threshold. The daily edges
hold one row per day, counterparty and rail with a count and a total; the size of a single payment is exact when the
count is 1 and an average otherwise.

In [ ]:
# =====================================================================
# 3 · SPLIT PAYMENTS — daily edges                                  [OUTPUT BLOCK 3]
# =====================================================================
t0 = time.time(); B = Block("§3 · Split payments near reporting thresholds")
SPLIT = None
if ED_OK:
    ED = spark.table(EDGE_DAILY).filter(F.col("month").isin(M_WIN))
    if "is_self" in ED.columns: ED = ED.filter(~F.coalesce(F.col("is_self"), F.lit(False)))
    side = lambda cust, cpty, ckind, d: ED.filter(F.col(f"{cust}_kind") == "CUST").select(
        "month", F.col("txn_date").cast("string").alias("txn_date"), F.col(f"{cust}_id").alias("mdm_id"), F.col(f"{cpty}_id").alias("cpty_id"),
        F.col(f"{cpty}_kind").alias("cpty_kind"), "category", F.col("amount").cast("double").alias("amount"), F.col("n_txn").cast("int").alias("n_txn"),
        F.lit(d).alias("direction"))
    E = side("src", "dst", "dst", "out").unionByName(side("dst", "src", "src", "in"))
    E = (E.join(SC.select("mdm_id").distinct(), "mdm_id")                                    # screened entities only
          .withColumn("thr", F.explode(F.array(*[F.lit(float(x)) for x in SPLIT_THRESHOLDS])))
          .withColumn("per_txn", F.col("amount") / F.greatest(F.col("n_txn"), F.lit(1)))
          .withColumn("below", (F.col("per_txn") >= (1 - SPLIT_BAND) * F.col("thr")) & (F.col("per_txn") < F.col("thr")))
          .withColumn("above", (F.col("per_txn") >= F.col("thr")) & (F.col("per_txn") < (1 + SPLIT_BAND) * F.col("thr")))
          .withColumn("split_day", F.col("below") & (F.col("n_txn") >= 2) & (F.col("amount") >= F.col("thr"))))
    E = E.filter("below or above").persist(DISK)
    MS = (E.groupBy("mdm_id", "thr", "month").agg(
            F.sum(F.when(F.col("below"), F.col("n_txn")).otherwise(0)).alias("n_below_m"),
            F.sum(F.when(F.col("above"), F.col("n_txn")).otherwise(0)).alias("n_above_m"),
            F.sum(F.when(F.col("below"), F.col("amount")).otherwise(0.0)).alias("amt_below_m"),
            F.sum(F.col("split_day").cast("int")).alias("split_days_m"))
            .withColumn("t", t_col("month") - F.lit(M0)))
    rows = []
    for m in M_SCORE:
        tm = ym_idx(m) - M0
        rows.append(MS.filter(F.col("t").between(tm - 2, tm)).groupBy("mdm_id", "thr")
                      .agg(*[F.sum(f"{c}_m").alias(c) for c in ["n_below", "n_above", "amt_below", "split_days"]])
                      .withColumn("month", F.lit(m)))
    S3 = reduce(lambda a, b: a.unionByName(b), rows)
    S3 = (S3.withColumn("heap", (F.col("n_below") + 1) / (F.col("n_above") + 1))
            .withColumn("split_z", (F.col("n_below") - F.col("n_above")) / F.sqrt(F.greatest(F.col("n_below") + F.col("n_above"), F.lit(1)))))
    wbest = Window.partitionBy("mdm_id", "month").orderBy(F.desc("heap"), F.desc("n_below"))
    SPLIT = (S3.withColumn("_r", F.row_number().over(wbest)).filter("_r = 1").drop("_r")
               .withColumnRenamed("thr", "split_thr")).persist(DISK)
    q = topd(SPLIT.groupBy("month").agg(F.count("*").alias("entities_with_near_threshold_payments"),
                                        F.sum((F.col("n_below") >= SPLIT_MIN_BELOW).cast("int")).alias(f"with_{SPLIT_MIN_BELOW}+_below"),
                                        F.sum(((F.col("n_below") >= SPLIT_MIN_BELOW) & (F.col("heap") >= SPLIT_MIN_RATIO)
                                               & (F.col("split_z") >= SPLIT_MIN_Z) & (F.col("split_days") >= SPLIT_MIN_DAYS)).cast("int")).alias("pass_gate"))
                  .orderBy("month"))
    B.table(q, "3a · Entities with payments near the thresholds, per scoring month (3-month windows)", save="f1_3a_split")
    allb = topd(E.withColumn("bucket", F.floor(F.col("per_txn") / (F.col("thr") * 0.02)) * 2)
                 .groupBy("thr", "bucket").agg(F.sum("n_txn").alias("payments")).orderBy("thr", "bucket"))
    B.table(allb, "3b · All screened entities: payments by size around each threshold (bucket = % of the threshold, 2-point bins)",
            save="f1_3b_heaping")
else:
    B.check("split payments", "INFO", "skipped: daily edges unavailable")
B.show(t0)

## §4 Round-trip flows (two-node cycles)

For each counterparty, money paid out and money received back over the 3-month window; the round-trip amount is the
smaller of the two. Payments to the client's own name at other banks and to other entities of the same corporate family
are excluded, since those are treasury and intercompany moves, not round-trips. Longer cycles (A → B → C → A) are a
graph search left for a later version.

In [ ]:
# =====================================================================
# 4 · ROUND-TRIP — counterparty pairs                               [OUTPUT BLOCK 4]
# =====================================================================
t0 = time.time(); B = Block("§4 · Round-trip flows")
RT, RT_PAIRS = None, None
if PAIR_OK:
    P = spark.table(PAIR_TABLE).filter(F.col("month").isin(M_WIN))
    if "scope" in P.columns: P = P.filter(F.col("scope").isin("int", "ext"))
    if "o_self" in P.columns: P = P.filter(~F.coalesce(F.col("o_self").cast("boolean"), F.lit(False)))
    P = (P.select(F.col("cust").alias("mdm_id"), "other", "month", F.col("dir"), F.col("amount").cast("double").alias("amount"),
                  *([F.col("scope")] if "scope" in P.columns else [F.lit("ext").alias("scope")]))
          .join(SC.select("mdm_id").distinct(), "mdm_id")
          .groupBy("mdm_id", "other", "scope", "month")
          .agg(F.sum(F.when(F.col("dir") == "in", F.col("amount")).otherwise(0.0)).alias("amt_in"),
               F.sum(F.when(F.col("dir") == "out", F.col("amount")).otherwise(0.0)).alias("amt_out"))
          .withColumn("t", t_col("month") - F.lit(M0)))
    fam = CD.select(F.col("mdm_id").alias("other"), F.col("master_pwr_id").alias("o_master")) if "master_pwr_id" in CD.columns else None
    own = CD.select("mdm_id", F.col("master_pwr_id").alias("my_master")) if "master_pwr_id" in CD.columns else None
    parts = []
    for m in M_SCORE:
        tm = ym_idx(m) - M0
        w = (P.filter(F.col("t").between(tm - 2, tm)).groupBy("mdm_id", "other", "scope")
              .agg(F.sum("amt_in").alias("amt_in"), F.sum("amt_out").alias("amt_out"))
              .filter((F.col("amt_in") > 0) & (F.col("amt_out") > 0))
              .withColumn("rt", F.least("amt_in", "amt_out")).withColumn("month", F.lit(m)))
        parts.append(w)
    PP = reduce(lambda a, b: a.unionByName(b), parts)
    if fam is not None:
        PP = (PP.join(own, "mdm_id", "left").join(fam, "other", "left")
                .filter(~((F.col("scope") == "int") & F.col("o_master").isNotNull() & (F.col("o_master") == F.col("my_master"))))
                .drop("o_master", "my_master"))
    RT_PAIRS = PP.filter(F.col("rt") >= RT_MIN_PAIR).persist(DISK)
    wtop = Window.partitionBy("mdm_id", "month").orderBy(F.desc("rt"))
    RT = (RT_PAIRS.withColumn("_r", F.row_number().over(wtop))
            .groupBy("mdm_id", "month").agg(F.sum("rt").alias("rt_amt"), F.count("*").alias("n_rt"),
                                            F.max(F.when(F.col("_r") == 1, F.col("other"))).alias("rt_top_other"),
                                            F.max(F.when(F.col("_r") == 1, F.col("rt"))).alias("rt_top_amt"))).persist(DISK)
    q = topd(RT.groupBy("month").agg(F.count("*").alias("entities_with_two_way_counterparties"),
                                     F.expr("percentile_approx(rt_amt, 0.5)").alias("median_round_trip"),
                                     F.sum((F.col("rt_amt") >= RT_MIN_AMT).cast("int")).alias(f"round_trip_over_{RT_MIN_AMT // 1000}k"))
              .orderBy("month"))
    B.table(q, "4a · Entities with money going out to and coming back from the same counterparties (3-month windows)",
            money=("median_round_trip",), save="f1_4a_roundtrip")
else:
    B.check("round-trip", "INFO", "skipped: pair table unavailable")
B.show(t0)

## §5 Peer comparison, typology scores, guards

**Peers** are entities in the same month, the same industry (3-digit NAICS; 2-digit or all industries when fewer than
100 entities) and the same size band (fifth of 3-month flow). A percentile of 0.95 means more extreme than 95% of peers.

| Typology | Gate (all must hold) | Score 0–100 |
|---|---|---|
| Pass-through | in ≈ out ≥ 90% over 3 months, ≥ 85% in 2 of 3 months; average balance ≤ 5 days of inflow (balance required) | 40% pass-through percentile, 40% turnover percentile, 20% flow percentile |
| Funnel | ≥ 20 payers a month, ≥ 10 payers per payee, largest payee ≥ 50% of money out, in ≈ out ≥ 70% | 35% fan-in percentile, 35% payee-concentration percentile, 30% ticket-size-ratio percentile |
| Split payments | ≥ 10 payments in the 10% band below a threshold, ≥ 3× as many below as above, the excess is significant (z ≥ 4), and ≥ 3 days where several under-threshold payments to one counterparty add up to more than the threshold | how lopsided (½), how many (0.3), split days (0.2) |
| Activity spike | month's flow ≥ $250k, ≥ 10× its usual month, ≥ 50% with new counterparties, not a new client | size of the jump (½), share with new counterparties (½) |
| Round-trip | ≥ $250k comes back, ≥ 25% of money out | starts at 50 when the rule is met; + up to 30 as the share that comes back rises from 25% to 75%; + up to 20 by flow percentile |

**Guards:** pass-through and funnel scores are multiplied by 0.5 for processor-like entities and 0.6 in industries where
pass-through is normal (0.3 when both hold). **Overall score** = highest typology score + 5 per additional typology (max 100). Tiers: High
≥ 70, Elevated ≥ 50, Watch below.

In [ ]:
# =====================================================================
# 5 · PEERS, SCORES, GUARDS                                         [OUTPUT BLOCK 5]
# =====================================================================
t0 = time.time(); B = Block("§5 · Typology scores")
X = SC
if SPLIT is not None: X = X.join(SPLIT.select("mdm_id", "month", "split_thr", "n_below", "n_above", "amt_below", "split_days", "heap", "split_z"),
                                ["mdm_id", "month"], "left")
else:
    for c in ["split_thr", "n_below", "n_above", "amt_below", "split_days", "heap", "split_z"]: X = X.withColumn(c, F.lit(None).cast("double"))
if RT is not None: X = X.join(RT, ["mdm_id", "month"], "left")
else:
    for c in ["rt_amt", "n_rt", "rt_top_amt"]: X = X.withColumn(c, F.lit(None).cast("double"))
    X = X.withColumn("rt_top_other", F.lit(None).cast("string"))
X = X.withColumn("rt_share", safe_div(F.coalesce(F.col("rt_amt"), F.lit(0.0)), F.col("out_raw_3m")))

# ── peer groups: industry (NAICS 3 → 2 → all) × size band ────────────
naics = F.regexp_replace(F.coalesce(F.col("naics_cd").cast("string"), F.lit("")), "[^0-9]", "")
X = (X.withColumn("n3", F.when(F.length(naics) >= 3, F.substring(naics, 1, 3)).otherwise("UNK"))
      .withColumn("n2", F.when(F.length(naics) >= 2, F.substring(naics, 1, 2)).otherwise("UNK"))
      .withColumn("size_band", F.ntile(5).over(Window.partitionBy("month").orderBy("flow_3m"))))
X = (X.withColumn("c3", F.count("*").over(Window.partitionBy("month", "n3", "size_band")))
      .withColumn("c2", F.count("*").over(Window.partitionBy("month", "n2", "size_band")))
      .withColumn("call", F.count("*").over(Window.partitionBy("month", "size_band")))
      .withColumn("peer_lvl", F.when((F.col("n3") != "UNK") & (F.col("c3") >= MIN_PEER), F.lit("n3"))
                               .when((F.col("n2") != "UNK") & (F.col("c2") >= MIN_PEER), F.lit("n2")).otherwise(F.lit("all")))
      .withColumn("peer_group", F.when(F.col("peer_lvl") == "n3", F.concat(F.lit("NAICS "), F.col("n3")))
                                 .when(F.col("peer_lvl") == "n2", F.concat(F.lit("NAICS "), F.col("n2"))).otherwise(F.lit("all industries")))
      .withColumn("peer_n", F.when(F.col("peer_lvl") == "n3", F.col("c3")).when(F.col("peer_lvl") == "n2", F.col("c2")).otherwise(F.col("call"))))
# percentiles within the chosen peer group; "all industries" = the whole size band (not only the leftovers of small industries);
# rows where the measure is missing are ranked separately so they do not push everyone else up
LVL = {"n3": ["month", "n3", "size_band"], "n2": ["month", "n2", "size_band"], "all": ["month", "size_band"]}
for c in ["pt_3m", "turnover", "flow_3m", "fan", "top1_out", "ticket_ratio", "rt_share"]:
    pr = {k: F.percent_rank().over(Window.partitionBy(*cols, F.col(c).isNull()).orderBy(F.col(c))) for k, cols in LVL.items()}
    X = X.withColumn(f"pr_{c}", F.when(F.col(c).isNotNull(), F.when(F.col("peer_lvl") == "n3", pr["n3"])
                                                              .when(F.col("peer_lvl") == "n2", pr["n2"]).otherwise(pr["all"])))
X = X.drop("c3", "c2", "call", "peer_lvl")

# ── guards ───────────────────────────────────────────────────────────
X = (X.withColumn("processor_like", (F.col("payers_avg") >= PROCESSOR_MIN_CPTY) | (F.col("payees_avg") >= PROCESSOR_MIN_CPTY))
      .withColumn("industry_pt", naics.rlike("^(" + "|".join(PASS_THROUGH_NAICS) + ")"))
      .withColumn("treasury_sweep", F.coalesce(F.col("selfpay_share_3m"), F.lit(0.0)) >= 0.5)
      .withColumn("guard", F.when(F.col("processor_like"), F.lit(DISCOUNT_PROCESSOR)).otherwise(F.lit(1.0))      # both apply when both hold
                         * F.when(F.col("industry_pt"), F.lit(DISCOUNT_INDUSTRY)).otherwise(F.lit(1.0))))
z = lambda c: F.coalesce(F.col(c), F.lit(0.0))
gate_pt = (F.col("pt_3m") >= PT_MIN) & (F.col("pt_months") >= PT_MONTHS_REQ) & F.col("days_held").isNotNull() & (F.col("days_held") <= PT_MAX_DAYS_HELD)
gate_fn = ((F.col("payers_avg") >= FUN_MIN_PAYERS) & (F.col("fan") >= FUN_MIN_FAN) & (F.col("top1_out") >= FUN_MIN_TOP1_OUT)
           & (F.col("pt_3m") >= FUN_MIN_PT))
gate_sp = (z("n_below") >= SPLIT_MIN_BELOW) & (z("heap") >= SPLIT_MIN_RATIO) & (z("split_z") >= SPLIT_MIN_Z) & (z("split_days") >= SPLIT_MIN_DAYS)
gate_sk = ((F.col("flow_m") >= SPIKE_MIN_FLOW_M) & (F.col("spike_x") >= SPIKE_MIN_X) & (z("new_share_m") >= SPIKE_MIN_NEW)
           & F.col("existed_before"))
gate_rt = (z("rt_amt") >= RT_MIN_AMT) & (z("rt_share") >= RT_MIN_SHARE)
X = (X.withColumn("score_passthrough", F.when(gate_pt, 100 * F.col("guard") * (0.4 * z("pr_pt_3m") + 0.4 * F.coalesce(F.col("pr_turnover"), z("pr_flow_3m"))
                                                                                + 0.2 * z("pr_flow_3m"))).otherwise(0.0))
      .withColumn("score_funnel", F.when(gate_fn, 100 * F.col("guard") * (0.35 * z("pr_fan") + 0.35 * z("pr_top1_out")
                                                                           + 0.30 * F.coalesce(F.col("pr_ticket_ratio"), F.lit(0.5)))).otherwise(0.0))
      .withColumn("score_split", F.when(gate_sp, 100 * F.least(F.lit(1.0), 0.5 * F.least(z("heap") / 10, F.lit(1.0))
                                                               + 0.3 * F.least(z("n_below") / 30, F.lit(1.0))
                                                               + 0.2 * F.least(z("split_days") / 5, F.lit(1.0)))).otherwise(0.0))
      .withColumn("score_spike", F.when(gate_sk, 100 * F.least(F.lit(1.0), 0.5 * F.least(F.log10("spike_x") / 2, F.lit(1.0))
                                                               + 0.5 * z("new_share_m"))).otherwise(0.0))
      .withColumn("score_roundtrip", F.when(gate_rt, 100 * (0.5 + 0.3 * F.least((z("rt_share") - RT_MIN_SHARE) / 0.5, F.lit(1.0))
                                                             + 0.2 * z("pr_flow_3m"))).otherwise(0.0)))
TYP = ["passthrough", "funnel", "split", "spike", "roundtrip"]
USD = {"passthrough": F.least("in_3m", "out_3m"), "funnel": F.col("in_3m"), "split": z("amt_below"), "spike": F.col("flow_m"), "roundtrip": z("rt_amt")}
X = X.withColumn("n_typ", sum((F.col(f"score_{k}") > 0).cast("int") for k in TYP))
X = X.withColumn("top_score", F.greatest(*[F.col(f"score_{k}") for k in TYP]))
X = (X.withColumn("score", F.when(F.col("n_typ") > 0, F.least(F.lit(100.0), F.col("top_score") + 5 * (F.col("n_typ") - 1))).otherwise(0.0))
      .withColumn("primary", F.coalesce(*[F.when(F.col(f"score_{k}") == F.col("top_score"), F.lit(k)) for k in TYP]))
      .withColumn("primary", F.when(F.col("n_typ") > 0, F.col("primary")))
      .withColumn("typologies", F.concat_ws(",", *[F.when(F.col(f"score_{k}") > 0, F.lit(k)) for k in TYP]))
      .withColumn("usd_involved", F.greatest(*[F.when(F.col(f"score_{k}") > 0, USD[k]).otherwise(F.lit(0.0)) for k in TYP]))
      .withColumn("tier", F.when(F.col("score") >= TIER_HIGH, "High").when(F.col("score") >= TIER_ELEVATED, "Elevated")
                           .when(F.col("score") > 0, "Watch")))
X = X.persist(DISK)
cnt = topd(X.groupBy("month").agg(F.count("*").alias("screened"),
                                  *[F.sum((F.col(f"score_{k}") > 0).cast("int")).alias(k) for k in TYP],
                                  F.sum((F.col("n_typ") > 0).cast("int")).alias("any"),
                                  F.sum((F.col("n_typ") > 1).cast("int")).alias("two_or_more"),
                                  F.sum((F.col("tier") == "High").cast("int")).alias("high"),
                                  F.sum((F.col("tier") == "Elevated").cast("int")).alias("elevated")).orderBy("month"))
B.table(cnt, "5a · Entities matching each typology, per scoring month", save="f1_5a_counts")
pg = topd(X.filter(F.col("month") == END).groupBy("peer_group").agg(F.count("*").alias("entities")).orderBy(F.desc("entities")))
B.table(pg.head(15), f"5b · Largest peer groups in {END} (each split into five size bands)")
B.show(t0)

## §6 What the flagged cases look like

The strongest cases per typology in the latest month, with the evidence the app shows on their cards. Names are shown
here for the analyst; the app shows them only to viewers allowed to see the fraud module.

In [ ]:
# =====================================================================
# 6 · TOP CASES PER TYPOLOGY                                        [OUTPUT BLOCK 6]
# =====================================================================
t0 = time.time(); B = Block(f"§6 · Strongest cases per typology, {END}")
EVID = {"passthrough": ["in_3m", "out_3m", "pt_3m", "pt_months", "days_held", "pr_pt_3m"],
        "funnel": ["payers_avg", "payees_avg", "fan", "top1_out", "ticket_in", "ticket_out", "wire_out"],
        "split": ["split_thr", "n_below", "n_above", "heap", "split_z", "split_days", "amt_below"],
        "spike": ["flow_m", "base", "spike_x", "new_share_m"],
        "roundtrip": ["rt_amt", "rt_share", "n_rt", "rt_top_amt"]}
L = X.filter(F.col("month") == END)
for k in TYP:
    cols = ["customer_name", "naics_desc", f"score_{k}", "score", "typologies", "processor_like", "industry_pt"] + EVID[k]
    cols = [c for c in cols if c in L.columns]
    d = topd(L.filter(F.col(f"score_{k}") > 0).orderBy(F.desc(f"score_{k}")).select(*cols).limit(12))
    B.table(d, f"6 · {k}", pct=tuple(c for c in ["pt_3m", "top1_out", "wire_out", "new_share_m", "rt_share", "pr_pt_3m"] if c in d.columns),
            money=tuple(c for c in ["in_3m", "out_3m", "amt_below", "flow_m", "base", "rt_amt", "rt_top_amt", "ticket_in", "ticket_out"] if c in d.columns),
            save=f"f1_6_{k}")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
last = cnt.iloc[-1]; ax[0].bar(TYP, [last[k] for k in TYP], color="#1d4e89"); ax[0].set_title(f"Entities per typology, {END}")
sd = topd(L.filter(F.col("score") > 0).select("score"))
ax[1].hist(sd.score, bins=20, color="#d1495b"); ax[1].set_title("Overall score of flagged entities")
for x in (TIER_ELEVATED, TIER_HIGH): ax[1].axvline(x, color="grey", ls="--")
savefig(fig, "f1_6_overview")
B.note(f"chart saved: {OUT / 'f1_6_overview.png'}")
B.show(t0)

## §7 Checks and the tables for the app

In [ ]:
# =====================================================================
# 7 · CHECKS AND WRITE                                              [OUTPUT BLOCK 7]
# =====================================================================
t0 = time.time(); B = Block("§7 · Checks and tables for the app")
last = cnt[cnt.month == END].iloc[0]
rates = {k: last[k] / max(last["screened"], 1) for k in TYP}
worst = max(rates, key=rates.get)
B.check("E1 · each typology flags < 2% of screened entities", "READY" if rates[worst] < 0.02 else "FIX",
        f"highest: {worst} at {rates[worst]:.2%}" + ("" if rates[worst] < 0.02 else " — tighten its gate in §0"))
g = L.filter(F.col("n_typ") > 0).agg(F.count("*").alias("n"), F.sum((F.col("processor_like") | F.col("industry_pt")).cast("int")).alias("g")).first()
gshare = (g["g"] or 0) / max(g["n"], 1)
B.check("E2 · < 30% of flagged are processor-like or in pass-through industries", "READY" if gshare < 0.30 else "FIX",
        f"{gshare:.1%} of {g['n']:,} flagged" + ("" if gshare < 0.30 else " — guards are not biting; raise the discounts"))
B.check("E3 · some entities match two or more typologies", "READY" if last["two_or_more"] > 0 else "INFO", f"{int(last['two_or_more']):,} in {END}")

# ── names for counterparties (external from the counterparty dimension, internal from the customer dimension) ──
def names_for(ids_df, col):
    ext = spark.table(CPTY_DIM).select(F.col("cpty_id").alias(col), F.col(CPTY_NAME_COL).alias("cpty_name"), F.col(CPTY_FI_COL).alias("bank"))
    intl = CD.select(F.col("mdm_id").alias(col), F.col("customer_name").alias("cpty_name"), F.lit("PNC").alias("bank"))
    ids = ids_df.select(col).distinct()
    return ids.join(intl, col, "left").join(ext.withColumnRenamed("cpty_name", "n2").withColumnRenamed("bank", "b2"), col, "left") \
              .select(col, F.coalesce("cpty_name", "n2").alias("cpty_name"), F.coalesce("bank", "b2").alias("bank"))

FLAG = X.filter(F.col("n_typ") > 0).select("mdm_id").distinct()
SIG = X.join(FLAG, "mdm_id")
if RT is not None:
    nm = names_for(SIG.select(F.col("rt_top_other")).filter("rt_top_other is not null"), "rt_top_other")
    SIG = SIG.join(nm.withColumnRenamed("cpty_name", "rt_top_name").drop("bank"), "rt_top_other", "left")
else:
    SIG = SIG.withColumn("rt_top_name", F.lit(None).cast("string"))
SIG_COLS = (["mdm_id", "customer_pwr_id", "pwr", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc", "month",
             "score", "tier", "primary", "typologies", "n_typ", "usd_involved"] + [f"score_{k}" for k in TYP]
            + ["processor_like", "industry_pt", "treasury_sweep", "existed_before",
               "in_3m", "out_3m", "out_raw_3m", "flow_3m", "pt_3m", "pt_months", "bal_3m", "days_held", "turnover",
               "payers_avg", "payees_avg", "fan", "top1_out", "ticket_in", "ticket_out", "ticket_ratio", "wire_out",
               "new_payer_share", "ext_in_share", "selfpay_share_3m", "flow_m", "base", "spike_x", "new_share_m",
               "split_thr", "n_below", "n_above", "amt_below", "split_days", "heap", "split_z",
               "rt_amt", "rt_share", "n_rt", "rt_top_other", "rt_top_name", "rt_top_amt",
               "peer_group", "size_band", "peer_n"] + [c for c in X.columns if c.startswith("pr_")])
SIG = SIG.select(*[F.col(c) for c in SIG_COLS if c in SIG.columns])
for c, t in SIG.dtypes:
    if t.startswith("decimal") or t in ("float",): SIG = SIG.withColumn(c, F.col(c).cast("double"))
write_full(SIG, f"{OUT_PREFIX}signals", 4)

HIST_COLS = ["mdm_id", "month", "amt_all_in", "amt_all_out", "out_adj", "ntxn_all_in", "ntxn_all_out", "ncpty_all_in", "ncpty_all_out",
             "top1_share_out", "selfpay_amt_out", "new_amt_share_in", "new_amt_share_out", "share_WIRE_out", "n_fi_out", "active_days_in",
             "bal_avg", "pt_m", "flow_m"]
HIST = G.join(FLAG, "mdm_id").select(*[c for c in HIST_COLS if c in G.columns])
write_full(HIST, f"{OUT_PREFIX}history", 4)

# ── evidence for the latest window: near-threshold payment days and round-trip counterparties ──
EV = []
EVCOLS = ["mdm_id", "typology", "month", "txn_date", "direction", "cpty_id", "cpty_name", "bank", "n_txn", "amount", "amount_back",
          "threshold", "split_day"]
win = M_HIST[-3:]
if SPLIT is not None:
    sp_ids = L.filter(F.col("score_split") > 0).select("mdm_id", F.col("split_thr").alias("thr"))
    se = (E.join(sp_ids, ["mdm_id", "thr"]).filter(F.col("below") & F.col("month").isin(win))
           .select("mdm_id", F.lit("split").alias("typology"), "month", "txn_date", "direction", "cpty_id", "n_txn", "amount",
                   F.lit(None).cast("double").alias("amount_back"), F.col("thr").alias("threshold"), "split_day"))
    se = se.join(names_for(se, "cpty_id"), "cpty_id", "left")
    EV.append(se.select(*EVCOLS))
if RT_PAIRS is not None:
    rt_ids = L.filter(F.col("score_roundtrip") > 0).select("mdm_id")
    re_ = (RT_PAIRS.filter(F.col("month") == END).join(rt_ids, "mdm_id")
             .select("mdm_id", F.lit("roundtrip").alias("typology"), "month", F.lit(None).cast("string").alias("txn_date"),
                     F.col("scope").alias("direction"), F.col("other").alias("cpty_id"), F.lit(None).cast("int").alias("n_txn"),
                     F.col("amt_out").alias("amount"), F.col("amt_in").alias("amount_back"), F.lit(None).cast("double").alias("threshold"),
                     F.lit(None).cast("boolean").alias("split_day")))
    re_ = re_.join(names_for(re_, "cpty_id"), "cpty_id", "left")
    EV.append(re_.select(*EVCOLS))
if EV:
    write_full(reduce(lambda a, b: a.unionByName(b), EV), f"{OUT_PREFIX}evidence", 2)
else:
    write_full(spark.createDataFrame([], "mdm_id string, typology string, month string, txn_date string, direction string, cpty_id string, "
                                         "cpty_name string, bank string, n_txn int, amount double, amount_back double, threshold double, split_day boolean"),
               f"{OUT_PREFIX}evidence", 1)

back = {t: spark.table(f"{TGT_DB}.{OUT_PREFIX}{t}").count() for t in ("signals", "history", "evidence")}
fl = spark.table(f"{TGT_DB}.{OUT_PREFIX}signals").filter(F.col("month") == END)
tiers = topd(fl.filter("score > 0").groupBy("tier").agg(F.count("*").alias("entities"), F.sum("usd_involved").alias("usd_involved")).orderBy("tier"))
B.table(tiers, f"7a · Flagged entities in {END} by tier", money=("usd_involved",), save="f1_7a_tiers")
B.kv([(f"{TGT_DB}.{OUT_PREFIX}signals", f"{back['signals']:,} rows (flagged entities × {N_SCORE} months)"),
      (f"{TGT_DB}.{OUT_PREFIX}history", f"{back['history']:,} rows ({HIST_MONTHS} months per flagged entity)"),
      (f"{TGT_DB}.{OUT_PREFIX}evidence", f"{back['evidence']:,} rows (near-threshold payment days and round-trip pairs, {win[0]}…{END})"),
      ("Impala, once per new table", " ".join(f"INVALIDATE METADATA {TGT_DB}.{OUT_PREFIX}{t};" for t in ("signals", "history", "evidence"))),
      ("wall time", f"{(time.time() - T0) / 60:,.1f} min")], "7b · Written")
B.table(pd.DataFrame(CHECKS), "7c · Checks")
for d_ in (G, SC, X): d_.unpersist()
B.show(t0)